# HMS merge

按章节合并 Lecture、按 Exercise 合并练习、按时间顺序合并全部考试；最后一个 cell 单独翻译所有合并后的 PDF。

In [1]:
from pathlib import Path
import re
from collections import defaultdict

HMS_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hms")
LECTURE_DIR = HMS_DIR / "Lecture Material"
EXERCISE_DIR = HMS_DIR / "Exercise Material"
EXAM_DIR = HMS_DIR / "Exam Preparation Material"
EXTS = {".ppt", ".pptx", ".pdf"}

def numeric_key(path):
    return tuple(int(x) for x in re.findall(r"\d+", path.stem))

def count_pdf_pages(path):
    from pypdf import PdfReader
    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    from pypdf import PdfWriter
    writer = PdfWriter()
    file_pages = []
    for path in files:
        pages = count_pdf_pages(path)
        file_pages.append((path, pages))
        writer.append(str(path))
    output_path.parent.mkdir(exist_ok=True)
    with output_path.open("wb") as output:
        writer.write(output)
    return file_pages, count_pdf_pages(output_path)

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32；该方法需要 Windows + PowerPoint。")
    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True
    merged = app.Presentations.Add()
    file_pages = []
    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()
        for path in files:
            src = app.Presentations.Open(str(path.resolve()), ReadOnly=True, WithWindow=False)
            pages = src.Slides.Count
            file_pages.append((path, pages))
            src.Close()
            if pages > 0:
                merged.Slides.InsertFromFile(str(path.resolve()), merged.Slides.Count, 1, pages)
        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))
    finally:
        merged.Close()
        app.Quit()
    return file_pages, total_pages

def merge_group(files, output_dir, group):
    files = sorted(files, key=numeric_key)
    ppt_files = [p for p in files if p.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [p for p in files if p.suffix.lower() == ".pdf"]
    output_dir.mkdir(exist_ok=True)
    if ppt_files:
        pages, total = merge_ppts(ppt_files, output_dir / f"{group}.pptx")
        print(f"{group}.pptx: {len(pages)} files, {total} slides")
    if pdf_files:
        pages, total = merge_pdfs(pdf_files, output_dir / f"{group}.pdf")
        print(f"{group}.pdf: {len(pages)} files, {total} pages")


## 1. Merge lecture material by chapter

In [2]:
groups = defaultdict(list)
for path in LECTURE_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in EXTS and not path.name.startswith("~$") and "_merged" not in path.parts:
        match = re.match(r"^HMS_WS2526_Chapter(\d{2})", path.stem, re.IGNORECASE)
        if match:
            groups[f"Chapter{match.group(1)}"].append(path)

for group in sorted(groups, key=lambda name: int(re.search(r"\d+", name).group())):
    print("\n" + "=" * 80)
    print(f"章节 {group}: {len(groups[group])} 个文件")
    merge_group(groups[group], LECTURE_DIR / "_merged", group)



章节 Chapter00: 1 个文件
Chapter00.pdf: 1 files, 20 pages

章节 Chapter01: 1 个文件
Chapter01.pdf: 1 files, 24 pages

章节 Chapter02: 1 个文件
Chapter02.pdf: 1 files, 41 pages

章节 Chapter03: 1 个文件
Chapter03.pdf: 1 files, 49 pages

章节 Chapter04: 1 个文件
Chapter04.pdf: 1 files, 31 pages

章节 Chapter05: 1 个文件
Chapter05.pdf: 1 files, 92 pages

章节 Chapter06: 1 个文件
Chapter06.pdf: 1 files, 157 pages

章节 Chapter07: 1 个文件
Chapter07.pdf: 1 files, 14 pages

章节 Chapter08: 1 个文件
Chapter08.pdf: 1 files, 28 pages

章节 Chapter09: 4 个文件
Chapter09.pdf: 4 files, 181 pages

章节 Chapter10: 1 个文件
Chapter10.pdf: 1 files, 53 pages

章节 Chapter11: 1 个文件
Chapter11.pdf: 1 files, 36 pages

章节 Chapter12: 1 个文件
Chapter12.pdf: 1 files, 16 pages

章节 Chapter13: 1 个文件
Chapter13.pdf: 1 files, 63 pages

章节 Chapter14: 1 个文件
Chapter14.pdf: 1 files, 48 pages

章节 Chapter15: 1 个文件
Chapter15.pdf: 1 files, 11 pages


In [2]:
# 只修改已合并 Chapter 文件的名字，不修改文件内容。
LECTURE_OUTPUT_DIR = LECTURE_DIR / "_merged"
chapter_titles = {}

for source_path in LECTURE_DIR.iterdir():
    if not source_path.is_file() or source_path.suffix.lower() not in EXTS:
        continue
    match = re.match(r"^HMS_WS2526_Chapter(\d{2})(?:_\d+)*_(.+)$", source_path.stem, re.IGNORECASE)
    if match:
        chapter_titles[f"Chapter{match.group(1)}"] = match.group(2)

for chapter, title in sorted(chapter_titles.items()):
    for prefix in ("", "trans-"):
        for suffix in (".pdf", ".pptx"):
            old_path = LECTURE_OUTPUT_DIR / f"{prefix}{chapter}{suffix}"
            new_path = LECTURE_OUTPUT_DIR / f"{prefix}{chapter}_{title}{suffix}"
            if old_path.exists() and not new_path.exists():
                old_path.rename(new_path)
                print(f"重命名：{old_path.name} -> {new_path.name}")
            elif new_path.exists():
                print(f"已存在：{new_path.name}")


重命名：Chapter00.pdf -> Chapter00_Motivation.pdf
重命名：trans-Chapter00.pdf -> trans-Chapter00_Motivation.pdf
重命名：Chapter01.pdf -> Chapter01_Introduction.pdf
重命名：trans-Chapter01.pdf -> trans-Chapter01_Introduction.pdf
重命名：Chapter02.pdf -> Chapter02_Hardware_Design_Flow.pdf
重命名：trans-Chapter02.pdf -> trans-Chapter02_Hardware_Design_Flow.pdf
重命名：Chapter03.pdf -> Chapter03_Design_Partition.pdf
重命名：trans-Chapter03.pdf -> trans-Chapter03_Design_Partition.pdf
重命名：Chapter04.pdf -> Chapter04_Modeling_Basics.pdf
重命名：trans-Chapter04.pdf -> trans-Chapter04_Modeling_Basics.pdf
重命名：Chapter05.pdf -> Chapter05_Hardware_Design_Languages.pdf
重命名：trans-Chapter05.pdf -> trans-Chapter05_Hardware_Design_Languages.pdf
重命名：Chapter06.pdf -> Chapter06_Other_Languages.pdf
重命名：trans-Chapter06.pdf -> trans-Chapter06_Other_Languages.pdf
重命名：Chapter07.pdf -> Chapter07_Comparative_Summary.pdf
重命名：trans-Chapter07.pdf -> trans-Chapter07_Comparative_Summary.pdf
重命名：Chapter08.pdf -> Chapter08_Simulation_Basics.pdf
重命名：trans-C

In [8]:
# 按章节顺序合并所有 Lecture，并为每个 Chapter 添加一级 PDF 目录。
from pypdf import PdfReader, PdfWriter

def chapter_file_key(path):
    match = re.match(r"^Chapter(\d{2})_", path.stem, re.IGNORECASE)
    return int(match.group(1)) if match else 9999

def chapter_outline_title(path):
    match = re.match(r"^(Chapter\d{2})_(.+)$", path.stem, re.IGNORECASE)
    return f"{match.group(1)}: {match.group(2).replace('_', ' ')}"

chapter_files = sorted([
    path for path in LECTURE_OUTPUT_DIR.glob("Chapter*.pdf")
    if re.match(r"^Chapter\d{2}_.+", path.stem, re.IGNORECASE)
    and not path.name.startswith("trans-")
], key=chapter_file_key)

if not chapter_files:
    raise FileNotFoundError(f"没有找到带名称的 Chapter PDF：{LECTURE_OUTPUT_DIR}")

all_chapters_output = LECTURE_OUTPUT_DIR / "HMS_All_Chapters.pdf"
writer = PdfWriter()
total_pages = 0
for path in chapter_files:
    pages = len(PdfReader(str(path)).pages)
    writer.append(str(path), outline_item=chapter_outline_title(path), import_outline=False)
    total_pages += pages
    print(f"{path.name} -> {pages} pages")

with all_chapters_output.open("wb") as output:
    writer.write(output)
print(f"合并完成：{all_chapters_output}")
print(f"Chapter 数量：{len(chapter_files)}，总页数：{total_pages}")


Chapter00_Motivation.pdf -> 20 pages
Chapter01_Introduction.pdf -> 24 pages
Chapter02_Hardware_Design_Flow.pdf -> 41 pages
Chapter03_Design_Partition.pdf -> 49 pages
Chapter04_Modeling_Basics.pdf -> 31 pages
Chapter05_Hardware_Design_Languages.pdf -> 92 pages
Chapter06_Other_Languages.pdf -> 157 pages
Chapter07_Comparative_Summary.pdf -> 14 pages
Chapter08_Simulation_Basics.pdf -> 28 pages
Chapter09_Simulation.pdf -> 181 pages
Chapter10_Power_Timing_Modeling.pdf -> 53 pages
Chapter11_Fault_Simulation.pdf -> 36 pages
Chapter12_Verification_Basics.pdf -> 16 pages
Chapter13_Simulation-based_Verification.pdf -> 63 pages
Chapter14_Formal_Verification.pdf -> 48 pages
Chapter15_Comparative_Summary.pdf -> 11 pages
合并完成：E:\OneDrive - MSFT\.master_data\25-26ws\hms\Lecture Material\_merged\HMS_All_Chapters.pdf
Chapter 数量：16，总页数：864


## 2. Merge exercise material by exercise number

In [3]:
def exercise_group(path):
    relative = path.relative_to(EXERCISE_DIR)
    text = " ".join(relative.parts)
    match = re.search(r"Exercise\s*0*(\d+)", text, re.IGNORECASE)
    if not match:
        match = re.search(r"(?:HMS_|Exercise[_ ])0*(\d+)(?:st|nd|rd|th)?", path.stem, re.IGNORECASE)
    return f"EX{int(match.group(1)):02d}" if match else None

def exercise_sort_key(path):
    name = path.stem.lower()
    role = 0 if any(word in name for word in ("exercise", "task", "setup")) else 1 if "solution" in name else 9
    return role, numeric_key(path), name

groups = defaultdict(list)
for path in EXERCISE_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in EXTS and not path.name.startswith("~$") and "_merged" not in path.parts:
        group = exercise_group(path)
        if group:
            groups[group].append(path)

for group in sorted(groups, key=lambda name: int(re.search(r"\d+", name).group())):
    print("\n" + "=" * 80)
    files = sorted(groups[group], key=exercise_sort_key)
    print(f"{group}: {len(files)} 个文件")
    merge_group(files, EXERCISE_DIR / "_merged", group)



EX01: 2 个文件
EX01.pdf: 2 files, 6 pages

EX02: 2 个文件
EX02.pdf: 2 files, 9 pages

EX03: 2 个文件
EX03.pdf: 2 files, 12 pages

EX04: 2 个文件
EX04.pdf: 2 files, 15 pages

EX05: 1 个文件
EX05.pdf: 1 files, 5 pages

EX06: 2 个文件
EX06.pdf: 2 files, 15 pages

EX07: 2 个文件
EX07.pdf: 2 files, 10 pages


## 3. Merge all exam preparation material

In [2]:
def exam_sort_key(path):
    text = path.stem.lower()
    match = re.search(r"(ws|ss)(\d{2,4})(\d{2})?", text)
    if match:
        digits = match.group(2)
        first = int(digits)
        if len(digits) == 4 and int(digits[2:]) == int(digits[:2]) + 1:
            year = 2000 + int(digits[:2])  # WS1516 -> 2015, WS1920 -> 2019
        else:
            year = first if first >= 1900 else 2000 + first
    else:
        years = [int(x) for x in re.findall(r"(?:19|20)\d{2}", text)]
        year = min(years, default=9999)
    term = 0 if "ss" in text else 1
    solution = 1 if "solution" in text else 0
    return year, term, solution, numeric_key(path), text

exam_files = sorted([
    path for path in EXAM_DIR.rglob("*.pdf")
    if path.is_file() and not path.name.startswith("~$") and "_merged" not in path.parts
], key=exam_sort_key)
if not exam_files:
    raise FileNotFoundError(f"没有找到考试 PDF: {EXAM_DIR}")

from pypdf import PdfReader, PdfWriter

exam_output = EXAM_DIR / "_merged" / "HMS_Exams.pdf"
def exam_period(path):
    match = re.search(r"(?:^|_)(WS|SS)(\d{2,4})", path.stem, re.IGNORECASE)
    return match.group(0).strip("_").upper() if match else path.stem

def exam_title(path):
    kind = "Solution" if "solution" in path.stem.lower() else "Task"
    return f"{exam_period(path)} - {kind}"

TASK_HEADING = re.compile(r"^(Task|Problem)\s+(\d+)(?![\d.])\s*:?[ \t]*(.*)$", re.IGNORECASE)

def extract_task_bookmarks(path):
    reader = PdfReader(str(path))
    bookmarks = []
    seen = set()
    for page_index, page in enumerate(reader.pages):
        lines = [" ".join(line.split()) for line in (page.extract_text() or "").splitlines() if line.strip()]
        page_entries = []
        for line in lines:
            # 部分 PDF 会把 Task 抽成 T ask。
            line = re.sub(r"\bT\s+ask\b", "Task", line, flags=re.IGNORECASE)
            line = re.sub(r"\bP\s+roblem\b", "Problem", line, flags=re.IGNORECASE)
            match = TASK_HEADING.match(line)
            if match and match.group(2) not in seen:
                page_entries.append((match.group(2), line))
        # 第一页通常是总目录，真正的 Task 从后面的页面开始。
        if page_index == 0 and len({number.split(".")[0] for number, _ in page_entries}) > 1:
            continue
        for number, title in page_entries:
            if number not in seen:
                seen.add(number)
                title = re.sub(r"\s+\d+(?:\s+\d+)*\s*$", "", title)
                bookmarks.append((number, title, page_index))
    return bookmarks

writer = PdfWriter()
total = 0
for path in exam_files:
    count = count_pdf_pages(path)
    page_start = total
    writer.append(str(path), import_outline=False)
    exam_node = writer.add_outline_item(exam_title(path), page_number=page_start)
    task_bookmarks = extract_task_bookmarks(path)
    for number, title, page_index in task_bookmarks:
        absolute_page = page_start + page_index
        writer.add_outline_item(title, page_number=absolute_page, parent=exam_node)
    total += count
    print(f"{path.name} -> {count} pages, {len(task_bookmarks)} task bookmarks")
exam_output.parent.mkdir(exist_ok=True)
with exam_output.open("wb") as output:
    writer.write(output)
print(f"HMS_Exams.pdf: {len(exam_files)} files, {total} pages")


WS1516 Test.pdf -> 27 pages, 9 task bookmarks
WS1516 Solution.pdf -> 27 pages, 9 task bookmarks
SS16 Test.pdf -> 25 pages, 8 task bookmarks
SS16 Solution.pdf -> 26 pages, 8 task bookmarks
WS1617 Test.pdf -> 25 pages, 8 task bookmarks
WS1617 Solution.pdf -> 27 pages, 8 task bookmarks
SS17 Test.pdf -> 29 pages, 8 task bookmarks
SS17 Solution_with_Hints.pdf -> 31 pages, 8 task bookmarks
WS1718 Test.pdf -> 31 pages, 8 task bookmarks
WS1718 Solution_with_Hints.pdf -> 33 pages, 8 task bookmarks
SS18 Test.pdf -> 32 pages, 8 task bookmarks
SS18 Solution.pdf -> 34 pages, 8 task bookmarks
SS18 Solution_with_Hints.pdf -> 35 pages, 8 task bookmarks
WS1819 Test.pdf -> 28 pages, 8 task bookmarks
WS1819 Solution.pdf -> 28 pages, 8 task bookmarks
SS19 Test.pdf -> 33 pages, 8 task bookmarks
SS2019 Solution.pdf -> 36 pages, 8 task bookmarks
WS1920 Test.pdf -> 32 pages, 8 task bookmarks
WS1920 Solution.pdf -> 35 pages, 8 task bookmarks
SS2021 Test.pdf -> 34 pages, 8 task bookmarks
SS2021 Solution.pdf -> 

## 4. Translate all merged PDFs (run separately after merging)

In [ ]:
from pathlib import Path
import subprocess
import tempfile
import shutil

OVERWRITE = False
pdf2zh_next_cmd = shutil.which("pdf2zh_next")
if pdf2zh_next_cmd is None:
    raise RuntimeError("没有找到 pdf2zh_next。请先运行：pip install pdf2zh-next")

FOLDERS = [LECTURE_DIR / "_merged", EXERCISE_DIR / "_merged", EXAM_DIR / "_merged"]
pdf_files = sorted([
    p for folder in FOLDERS for p in folder.glob("*.pdf")
    if p.is_file()
    and not p.name.startswith("trans-")
    and not p.name.startswith("~$")
])
print("=" * 80)
print(f"待翻译文件夹：{FOLDERS}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("trans-" + pdf_path.name)
    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue
    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)
        cmd = [
            pdf2zh_next_cmd,
            str(tmp_input),
            "--lang-in", "en",
            "--lang-out", "zh-CN",
            "--no-mono",
            # 翻译论文要把这俩注释掉，因为 PDF 不能断行，不然翻译不连续；PPT 短句子要开
            "--split-short-lines",
            "--short-line-split-factor", "2.0",
            # "--disable-rich-text-translate",
            "--ignore-cache",
            "--watermark-output-mode", "no_watermark",
            # "--enhance-compatibility",
            # "--use-alternating-pages-dual",
            
            # 提速
            # "--qps", "20",
            # "--pool-max-workers", "100",

            # # 如果不需要自动术语提取，也建议关闭
            # "--no-auto-extract-glossary",
        ]
        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )
        print(result.stdout[-2000:])
        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))
        if not candidates:
            candidates = list(tmpdir.glob("*.pdf"))
            candidates = [p for p in candidates if p.name != pdf_path.name]
        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue
        if out_path.exists() and OVERWRITE:
            out_path.unlink()
        translated_pdf = candidates[0]
        shutil.move(str(translated_pdf), str(out_path))
        print(f"完成：{out_path.name}")
print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)


待翻译文件夹：[WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Lecture Material/_merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Exercise Material/_merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Exam Preparation Material/_merged')]
待翻译 PDF 数量：24

--------------------------------------------------------------------------------
开始翻译：HMS_Exams.pdf
DetectScannedFile (1/1)                                ----- 674/… 0:00… 0:00:…
Parse Page Layout (1/1)                                ----- 1348… 0:08… 0:00:…
Parse Paragraphs (1/1)                                 ----- 674/… 0:00… 0:00:…
Parse Formulas and Styles (1/1)                        ----- 674/… 0:00… 0:00:…
Automatic Term Extraction (1/1)                        ----- 2350… 0:02… 0:00:…
Translate Paragraphs (1/1)                             ----- 2350… 0:05… 0:00:…
Typesetting (1/1)                                      ----- 1348… 0:00… 0:00:…
Add Fonts (1/1)                                     